# Sports Card Extractor (Enhanced)

Extract structured data from sports card images using OpenAI's vision API with **dual-image support** (front + back).

## Features
- 📁 **Local file upload** - Upload images from your computer
- 🔄 **Front + Back processing** - Extract from both sides simultaneously
- 🎯 **Comprehensive extraction** - Brand, year, player, card #, serial, auto, parallels, visual descriptors
- 🔍 **eBay search generation** - Ready-to-use search URLs with visual keywords

## 1. Setup & Install

In [ ]:
!pip install openai python-dotenv pydantic pandas ipywidgets --quiet

In [ ]:
import os
import base64
import webbrowser
from pathlib import Path
from urllib.parse import quote
from typing import Optional, List

import openai
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from IPython.display import display, Image, Markdown, HTML
import ipywidgets as widgets
from IPython.display import clear_output

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

## 2. Data Model (Pydantic)

Comprehensive schema with **visual_keywords** for searchable visual terms.

In [ ]:
class SportsCard(BaseModel):
    """Comprehensive structured representation of a sports card."""

    # === CORE IDENTIFICATION ===
    player_name: str = Field(
        description="Full name of the player (e.g. 'Aaron Judge', 'Patrick Mahomes II')"
    )
    team: Optional[str] = Field(
        default=None,
        description="Team name shown on card"
    )
    position: Optional[str] = Field(
        default=None,
        description="Player position (e.g. 'OF', 'QB', 'PG')"
    )
    sport: Optional[str] = Field(
        default=None,
        description="Sport: Baseball, Football, Basketball, Hockey, Soccer"
    )
    
    # === CARD DETAILS ===
    manufacturer: Optional[str] = Field(
        default=None,
        description="Card manufacturer (Topps, Panini, Upper Deck, Bowman, Donruss)"
    )
    product_set: Optional[str] = Field(
        default=None,
        description="Product/Set name (Chrome, Prizm, Finest, Optic, Mosaic, Select)"
    )
    year: Optional[int] = Field(
        default=None,
        description="Year from COPYRIGHT line only (e.g., © 2025)"
    )
    card_number: Optional[str] = Field(
        default=None,
        description="Card number from back of card"
    )
    
    # === RARITY & VARIANTS ===
    serial_number: Optional[str] = Field(
        default=None,
        description="Serial numbering (e.g. '25/99', '1/1')"
    )
    parallel_type: Optional[str] = Field(
        default=None,
        description="EXACT parallel name using official terminology (see KNOWN PARALLELS list)"
    )
    insert_name: Optional[str] = Field(
        default=None,
        description="Insert set name if not base card"
    )
    
    # === SPECIAL FEATURES ===
    is_rookie_card: Optional[bool] = Field(
        default=None,
        description="True if marked as rookie card (RC logo)"
    )
    is_autograph: Optional[bool] = Field(
        default=None,
        description="True if card contains an autograph"
    )
    is_memorabilia: Optional[bool] = Field(
        default=None,
        description="True if card contains jersey/patch/relic"
    )
    
    # === GRADING ===
    grading_company: Optional[str] = Field(
        default=None,
        description="Grading company (PSA, BGS, SGC, CGC)"
    )
    grade: Optional[str] = Field(
        default=None,
        description="Grade value (10, 9.5, GEM MT 10)"
    )
    
    # === VISUAL SEARCH TERMS ===
    visual_keywords: Optional[List[str]] = Field(
        default=None,
        description="List of SEARCHABLE visual terms from known patterns: colors (Gold, Blue, Green, Purple, Orange, Red, Black, Pink, Sky Blue, Magenta), patterns (Refractor, X-Fractor, Checkerboard, Wave, Cracked Ice, Prizm, Mosaic, Oil Spill, Disco, Lazer, Pulsar), finishes (Holo, Foil, Chrome)"
    )
    
    # === ADDITIONAL ===
    visual_description: Optional[str] = Field(
        default=None,
        description="Free-form visual description for reference"
    )
    additional_text: Optional[str] = Field(
        default=None,
        description="Other identifying text (SP, SSP, Photo Variation, Award mentions)"
    )

    @property
    def display_name(self) -> str:
        parts = []
        if self.year:
            parts.append(str(self.year))
        if self.manufacturer:
            parts.append(self.manufacturer)
        if self.product_set:
            parts.append(self.product_set)
        parts.append(self.player_name)
        if self.parallel_type:
            parts.append(f"- {self.parallel_type}")
        if self.is_rookie_card:
            parts.append("RC")
        if self.serial_number:
            parts.append(f"[{self.serial_number}]")
        return " ".join(parts)

    @property
    def ebay_search_query(self) -> str:
        """Build optimized eBay search with visual keywords."""
        tokens = []
        if self.year:
            tokens.append(str(self.year))
        if self.manufacturer:
            tokens.append(self.manufacturer)
        if self.product_set:
            tokens.append(self.product_set)
        tokens.append(self.player_name)
        if self.card_number:
            tokens.append(f"#{self.card_number}")
        
        # Add parallel type (the official name)
        if self.parallel_type:
            tokens.append(self.parallel_type)
        # Also add visual keywords for better matching
        elif self.visual_keywords:
            # Join keywords into search
            tokens.extend(self.visual_keywords[:3])  # Top 3 keywords
        
        if self.serial_number:
            parts = self.serial_number.split('/')
            if len(parts) == 2:
                tokens.append(f"/{parts[1]}")
        if self.is_rookie_card:
            tokens.append("RC")
        if self.is_autograph:
            tokens.append("Auto")
        if self.is_memorabilia:
            tokens.append("Relic")
        if self.grade:
            company = self.grading_company or "PSA"
            tokens.append(f"{company} {self.grade}")
        return " ".join(tokens)

    @property
    def ebay_url(self) -> str:
        return f"https://www.ebay.com/sch/i.html?_nkw={quote(self.ebay_search_query)}"

    def to_markdown_table(self) -> str:
        lines = ["| **Field** | **Value** |", "| :-- | :-- |"]
        for field_name, value in self.model_dump().items():
            if value is not None and value != [] and value != False:
                label = field_name.replace("_", " ").title()
                if isinstance(value, bool):
                    value = "✅ Yes"
                if isinstance(value, list):
                    value = ", ".join(str(v) for v in value)
                lines.append(f"| {label} | {value} |")
        return "\n".join(lines)

    def _repr_markdown_(self):
        return f"### {self.display_name}\n\n{self.to_markdown_table()}"

## 3. Image Encoding

In [ ]:
def encode_uploaded_file(file_info: dict) -> dict:
    """Convert uploaded file content to API-ready format."""
    content = file_info['content']
    name = file_info.get('name', 'image.jpg')
    ext = os.path.splitext(name)[1].lower().lstrip(".")
    mime = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "webp": "webp", "gif": "gif"}.get(ext, "jpeg")
    
    if isinstance(content, memoryview):
        content = bytes(content)
    
    b64 = base64.b64encode(content).decode("utf-8")
    return {
        "type": "image_url",
        "image_url": {"url": f"data:image/{mime};base64,{b64}"},
    }

## 4. Extraction Function

In [ ]:
SYSTEM_PROMPT = """
You are an expert sports card analyst. Extract structured information from the card images.

EXTRACTION PRIORITIES:
1. MANUFACTURER - Topps, Panini, Upper Deck, Bowman, Donruss, Fleer
2. PRODUCT SET - Chrome, Prizm, Finest, Optic, Mosaic, Select, etc.
3. YEAR - **ONLY from COPYRIGHT line** (e.g., "© 2025 Topps"). IGNORE years in stats.
4. PLAYER NAME
5. CARD NUMBER - Usually on back
6. SERIAL NUMBER - Format: XX/YY
7. AUTOGRAPH - Signature on front
8. PARALLEL TYPE - Use EXACT official name from KNOWN PARALLELS list below
9. VISUAL KEYWORDS - Extract searchable terms from what you see

=== KNOWN PARALLELS (use these EXACT names) ===

TOPPS FINEST:
- Refractor (base shimmer)
- X-Fractor (X pattern)
- Checkerboard Refractor (checkerboard pattern)
- Oil Spill Refractor (rainbow oil slick)
- Sky Blue Refractor, Purple Refractor, Blue Refractor, Green Refractor
- Gold Refractor, Orange Refractor, Black Refractor, Red Refractor
- Purple X-Fractor, Blue X-Fractor
- Purple Checkerboard Refractor, Blue Checkerboard Refractor
- Pearl Checkerboard Refractor
- SuperFractor (1/1)

TOPPS CHROME:
- Refractor, X-Fractor, Sepia Refractor
- Gold Refractor, Blue Refractor, Green Refractor, Purple Refractor
- Orange Refractor, Red Refractor, Black Refractor
- Pink Refractor, Aqua Refractor, Camo Refractor
- Rainbow Foil, Gold Wave
- SuperFractor (1/1)

PANINI PRIZM:
- Silver Prizm (base shimmer)
- Blue Prizm, Green Prizm, Red Prizm, Orange Prizm, Purple Prizm
- Gold Prizm, Black Prizm, Pink Prizm
- Blue Wave, Green Wave, Red Wave, Orange Wave
- Cracked Ice (shattered ice pattern)
- Disco (sparkle pattern)
- Lazer (beam pattern)
- Pulsar (glow effect)
- Snakeskin (scale pattern)
- Mosaic (tile pattern)
- No Huddle (football specific)
- Hyper, Ice, Neon
- Black & Gold, Red White Blue

UPPER DECK:
- Clear Cut (acetate/translucent)
- High Gloss
- Exclusives, UD Exclusives
- Outburst, Silver Outburst, Gold Outburst, Red Outburst

=== VISUAL KEYWORDS ===
Extract these SEARCHABLE terms if you see them:

COLORS: Gold, Blue, Green, Purple, Orange, Red, Black, Pink, Sky Blue, Magenta, Aqua, Teal, Silver

PATTERNS:
- Refractor (rainbow shimmer when tilted)
- X-Fractor (X or cross pattern in background)
- Checkerboard (checkered square pattern)
- Wave (flowing wave lines)
- Cracked Ice (shattered/fractured ice look)
- Oil Spill (rainbow/oil slick swirls)
- Disco (sparkle dots)
- Lazer/Laser (beam lines)
- Mosaic (tile/fragment pattern)
- Pulsar (glowing effect)
- Snakeskin (scale texture)
- Geometric (angular shapes)

FINISHES: Holo, Holographic, Foil, Chrome, Shimmer, Prizm

=== OUTPUT INSTRUCTIONS ===
- parallel_type: Use EXACT name from KNOWN PARALLELS (e.g., "Checkerboard Refractor" NOT "checkered")
- visual_keywords: List of 2-5 searchable terms (e.g., ["Checkerboard", "Refractor", "Holographic"])
- If you see a checkered pattern on a Topps Finest card, it's "Checkerboard Refractor"
- If you see an X pattern, it's "X-Fractor"
- REMEMBER: Card year = Copyright year (© 2025), NOT stats year (2024 season)
"""


def extract_card(front_file: dict = None, back_file: dict = None) -> SportsCard:
    """Extract card data from uploaded front and/or back images."""
    content_parts = [{"type": "text", "text": "Extract all structured data from this sports card."}]
    
    if front_file:
        content_parts.append({"type": "text", "text": "FRONT OF CARD:"})
        content_parts.append(encode_uploaded_file(front_file))
    
    if back_file:
        content_parts.append({"type": "text", "text": "BACK OF CARD:"})
        content_parts.append(encode_uploaded_file(back_file))
    
    if len(content_parts) == 1:
        raise ValueError("Must provide at least one image")
    
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content_parts},
        ],
        response_format=SportsCard,
        temperature=0.1,
    )

    card = completion.choices[0].message.parsed
    if card is None:
        raise ValueError(f"Extraction failed: {completion.choices[0].message.refusal}")
    return card

## 5. Upload Your Card Images

In [ ]:
# Create upload widgets
front_upload = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Front of Card'
)

back_upload = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Back of Card'
)

extract_button = widgets.Button(
    description='Extract Card Data',
    button_style='success',
    icon='search'
)

output = widgets.Output()
extracted_card = None

def on_extract_click(b):
    global extracted_card
    with output:
        clear_output()
        
        front_data = None
        back_data = None
        
        if front_upload.value:
            if isinstance(front_upload.value, tuple):
                front_data = front_upload.value[0]
            elif isinstance(front_upload.value, dict):
                front_data = list(front_upload.value.values())[0]
            print(f"✓ Front image loaded")
        
        if back_upload.value:
            if isinstance(back_upload.value, tuple):
                back_data = back_upload.value[0]
            elif isinstance(back_upload.value, dict):
                back_data = list(back_upload.value.values())[0]
            print(f"✓ Back image loaded")
        
        if not front_data and not back_data:
            print("⚠️ Please upload at least one image")
            return
        
        print("\n🔍 Extracting card data...\n")
        
        try:
            extracted_card = extract_card(front_file=front_data, back_file=back_data)
            display(extracted_card)
            print(f"\n📝 eBay Search Query: {extracted_card.ebay_search_query}")
            print(f"\n🔗 eBay URL: {extracted_card.ebay_url}")
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()

extract_button.on_click(on_extract_click)

print("📤 Upload your card images:")
print("   - Front: parallel type, autograph, visual")
print("   - Back: year (copyright), card number")
print("")
display(widgets.VBox([
    widgets.HBox([widgets.Label("Front:"), front_upload]),
    widgets.HBox([widgets.Label("Back: "), back_upload]),
    extract_button,
    output
]))

## 6. View Extracted Data

In [ ]:
if extracted_card:
    print("=== EXTRACTED DATA ===")
    print(f"Player: {extracted_card.player_name}")
    print(f"Year: {extracted_card.year}")
    print(f"Brand: {extracted_card.manufacturer} {extracted_card.product_set}")
    print(f"Card #: {extracted_card.card_number}")
    print(f"Parallel: {extracted_card.parallel_type}")
    print(f"Visual Keywords: {extracted_card.visual_keywords}")
    print(f"\n🔍 Search Query: {extracted_card.ebay_search_query}")
else:
    print("No card extracted yet. Upload images above.")

In [ ]:
if extracted_card:
    print(extracted_card.model_dump_json(indent=2))

## 7. Open eBay Search

In [ ]:
if extracted_card:
    print(f"Search query: {extracted_card.ebay_search_query}")
    display(Markdown(f"**[🔍 Search eBay for this card]({extracted_card.ebay_url})**"))

## 8. Known Parallel Reference

### Topps Finest (2024-2025)
| Pattern | Description |
|---------|-------------|
| Refractor | Base shimmer |
| X-Fractor | X/cross pattern |
| Checkerboard Refractor | Checkered squares |
| Oil Spill Refractor | Rainbow oil slick |
| Pearl Checkerboard | Pearl + checkerboard |
| SuperFractor | 1/1 |

### Panini Prizm
| Pattern | Description |
|---------|-------------|
| Silver Prizm | Base shimmer |
| Cracked Ice | Shattered ice |
| Wave | Flowing lines |
| Disco | Sparkle dots |
| Lazer | Beam lines |
| Snakeskin | Scale texture |
| Mosaic | Tile fragments |